In [1]:
# Imports
import os
import json
from typing import TypedDict, List

# Image
import base64
from io import BytesIO
from PIL import Image

# Audio
from pydub import AudioSegment
from pydub.playback import play

from openai import OpenAI

import gradio as gr

In [2]:
# Configs

# LLM Client
openai = OpenAI()

# LLM models
OPEN_AI_MODEL = "gpt-4o-mini"
OPEN_AI_IMAGE_MODEL = "dall-e-3"
OPEN_AI_AUDIO_MODEL = "tts-1"

## LLM Hyper parameters
IMAGE_SIZE = "1024x1024"
N = 1
RESPONSE_FORMAT = "b64_json"
VOICE = "onyx"
# VOICE = "alloy"
AUDIO_FORMAT = "mp3"

# LLM Instructions
system_message = "You are a helpful assistant for an Airline called FlightAI. \
    Give short, courteous answers, no more than 1 sentence. \
        Always be accurate. If you don't know the answer, say so."

city = "New York City"

# Tools
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city. Call this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": price_function}]

In [3]:
class ChatMessage(TypedDict):
    role: str
    content: str


def get_ticket_price(destination: str):
    return ticket_prices.get(destination.lower(), "Unknown")


def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)
    city = arguments.get("destination")
    price = get_ticket_price(city)
    response = {
        "role": "tool",
        "content": json.dumps({"destination": city, "price": price}),
        "tool_call_id": tool_call.id,
    }
    return response, city


def chat(message: str, history: List[ChatMessage]) -> str:
    messages = [{"role": "system", "content": system_message}]
    if history:
        messages.extend(history)
    messages.append({"role": "user", "content": message})

    response = openai.chat.completions.create(
        model=OPEN_AI_MODEL, messages=messages, tools=tools
    )

    if response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(
            model=OPEN_AI_MODEL, messages=messages
        )

    return response.choices[0].message.content

In [4]:
def artist(city):
    image_response = openai.images.generate(
        model = OPEN_AI_IMAGE_MODEL,
        prompt=f"An image representing a vacation in {city} showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
        size=IMAGE_SIZE,
        response_format=RESPONSE_FORMAT,
    )

    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

# image = artist(city=city)
# display(image)

In [5]:
def talker(message):
    response = openai.audio.speech.create(
        model=OPEN_AI_AUDIO_MODEL,
        voice=VOICE,
        input=message
    )

    audio_stream = BytesIO(response.content)
    audio = AudioSegment.from_file(audio_stream, format=AUDIO_FORMAT)
    play(audio)

talker ("I am filip and I have doubts about the math.")

Input #0, wav, from '/tmp/tmpp9g9uc4i.wav':   0KB sq=    0B f=0/0   
  Duration: 00:00:02.66, bitrate: 384 kb/s
  Stream #0:0: Audio: pcm_s16le ([1][0][0][0] / 0x0001), 24000 Hz, 1 channels, s16, 384 kb/s


In [6]:
def chat(history: List[ChatMessage]):
    messages = [{"role": "system", "content": system_message}]
    messages.extend(history)
    response = openai.chat.completions.create(
        model=OPEN_AI_MODEL, messages=messages, tools=tools
    )
    image = None

    if response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        response, city = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        image = artist(city)
        response = openai.chat.completions.create(
            model=OPEN_AI_MODEL, messages=messages
        )

    reply = response.choices[0].message.content
    history += [{"role": "assistant", "content": reply}]
    
    talker(reply)
    return history, image

# Example usage to demonstrate the chat function with the flight ticket price tool
user_message = f"Please provide me ticket price for a return ticket to London from {city}."
history = [{"role": "user", "content": user_message}]
history, image = chat(history)
print(history)

Input #0, wav, from '/tmp/tmp1z8ozukn.wav':   0KB sq=    0B f=0/0   
  Duration: 00:00:04.80, bitrate: 384 kb/s
  Stream #0:0: Audio: pcm_s16le ([1][0][0][0] / 0x0001), 24000 Hz, 1 channels, s16, 384 kb/s



[{'role': 'user', 'content': 'Please provide me ticket price for a return ticket to London from New York City.'}, {'role': 'assistant', 'content': 'The price for a return ticket from New York City to London is $799.'}]


In [7]:
with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500)
    with gr.Row():
        entry = gr.Textbox(label="Chat with our AI Assistant")
    with gr.Row():
        clear = gr.Button("Clear")

    def do_entry(message, history):
        history += [{"role": "user", "content": message}]
        return "", history
    
    entry.submit(fn=do_entry, inputs=[entry, chatbot], outputs=[entry, chatbot]).then(
        fn=chat, inputs=chatbot, outputs=[chatbot, image_output]
    )

    clear.click(fn=lambda: None, inputs=None, outputs=chatbot, queue=False)

ui.launch(inbrowser=True, share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://46fbd94543b3538354.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
